# SDXL Packaging-Domain LoRA Training (Google Colab Pro)

**Goal:** Train a uniform-caption style LoRA that teaches SDXL the "Indian snack packaging" visual domain. The trained LoRA will pair with IP-Adapter-Plus at inference to produce folk-art-conditioned packaging.

**Trigger token:** `ipsnackpkg` (used in both training captions and inference prompts to invoke the learned domain)

**Training set:** ~270 processed packaging images from Open Food Facts (CC-BY-SA), uploaded to your Google Drive.

**Expected runtime:** ~2 hours on A100, ~4 hours on L4. Output: a `.safetensors` LoRA file ~150–250 MB you'll download for local inference.

**Before running:** in Colab menu → Runtime → Change runtime type → **GPU: A100** (or L4 if A100 unavailable). Then Runtime → Connect.

## Cell 1 — Check GPU and install dependencies

If you don't see A100 / L4 / T4 in the output, change the runtime type and reconnect before continuing.

In [3]:
# Force-pin Pillow to a version compatible with the pinned transformers
# Also pin a few transitive deps that Colab's default versions break
!pip install -q --force-reinstall \
    "Pillow>=10.2,<11.0" \
    "numpy<2.0"

print("Pillow pinned. NOW: Runtime menu → Restart session → then run all cells again.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 129.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 142.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07


In [2]:
# Pin versions known to work together. Don't loosen these without testing.
!pip install -q --upgrade \
    "diffusers>=0.30,<0.32" \
    "transformers>=4.40,<4.50" \
    "accelerate>=0.30" \
    "peft>=0.11" \
    "bitsandbytes>=0.43" \
    "safetensors" \
    "datasets" \
    "wandb" \
    "Pillow>=10.2,<11.0" \
    "numpy<2.0"

# Verify
import diffusers, transformers, peft, accelerate
print("diffusers   :", diffusers.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("accelerate  :", accelerate.__version__)

diffusers   : 0.31.0
transformers: 4.49.0
peft        : 0.19.1
accelerate  : 1.13.0


## Cell 2 — Mount Drive and prepare training data

**One-time setup before running this notebook:** upload your `data/processed/packaging/` folder to Google Drive. Recommended location: `MyDrive/dissertation/data/processed/packaging/`. You can do this via the Drive web interface (drag-and-drop the folder).

Also upload `data/splits/train.csv` to `MyDrive/dissertation/data/splits/`.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/dissertation")
assert DRIVE_ROOT.exists(), f"Upload your dissertation folder to {DRIVE_ROOT} first"

DRIVE_IMAGES = DRIVE_ROOT / "data" / "processed" / "packaging"
DRIVE_SPLIT  = DRIVE_ROOT / "data" / "splits" / "train.csv"
DRIVE_LORA_OUT = DRIVE_ROOT / "outputs" / "lora_checkpoints"
DRIVE_LORA_OUT.mkdir(parents=True, exist_ok=True)

print("Images on Drive    :", DRIVE_IMAGES, "-", len(list(DRIVE_IMAGES.glob('*.png'))) if DRIVE_IMAGES.exists() else "MISSING")
print("Train CSV on Drive :", DRIVE_SPLIT, "-", "exists" if DRIVE_SPLIT.exists() else "MISSING")
print("LoRA output dir    :", DRIVE_LORA_OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Images on Drive    : /content/drive/MyDrive/dissertation/data/processed/packaging - 311
Train CSV on Drive : /content/drive/MyDrive/dissertation/data/splits/train.csv - exists
LoRA output dir    : /content/drive/MyDrive/dissertation/outputs/lora_checkpoints


In [4]:
# Copy images to Colab's local SSD for fast disk access during training.
# Reading from Drive in the training loop is ~10x slower.
import shutil, time

LOCAL_IMAGES = Path("/content/training_images")
LOCAL_IMAGES.mkdir(exist_ok=True)

t0 = time.time()
count = 0
for src in DRIVE_IMAGES.glob("*.png"):
    dst = LOCAL_IMAGES / src.name
    if not dst.exists():
        shutil.copy2(src, dst)
        count += 1
print(f"Copied {count} new files in {time.time()-t0:.1f}s")
print(f"Local image count: {len(list(LOCAL_IMAGES.glob('*.png')))}")

Copied 311 new files in 109.6s
Local image count: 311


In [5]:
# Build a metadata.jsonl file pairing each training image with the uniform caption.
# This format is what HuggingFace datasets expects for ImageFolder-with-metadata.
import json, pandas as pd

TRIGGER = "ipsnackpkg"
CAPTION_TEMPLATE = (
    "a photograph of {trigger}, an Indian snack packet, "
    "product photography on a white background"
).format(trigger=TRIGGER)
print("Caption used for all training images:")
print(f"  {CAPTION_TEMPLATE}\n")

# Load split (assumes 'image_id' or 'filepath' column from your preprocess script)
df = pd.read_csv(DRIVE_SPLIT)
print("Split CSV columns:", list(df.columns))
print(f"Rows: {len(df)}")

# Build jsonl in the format datasets.ImageFolder expects
metadata_path = LOCAL_IMAGES / "metadata.jsonl"
n_written = 0
with open(metadata_path, "w") as f:
    for _, row in df.iterrows():
        # Pull the filename from whichever column carries it
        candidate = row.get("image_id", "")
        if candidate:
            file_name = f"{candidate}.png"
        else:
            file_name = Path(row["filepath"]).name
        if not (LOCAL_IMAGES / file_name).exists():
            continue
        f.write(json.dumps({"file_name": file_name, "text": CAPTION_TEMPLATE}) + "\n")
        n_written += 1

print(f"\nmetadata.jsonl written: {n_written} usable training rows")
if n_written < 100:
    print("WARNING: fewer than 100 training samples. Check that image_id values in train.csv match the .png filenames.")

Caption used for all training images:
  a photograph of ipsnackpkg, an Indian snack packet, product photography on a white background

Split CSV columns: ['image_id', 'filepath', 'kind', 'tradition', 'original_path']
Rows: 280

metadata.jsonl written: 271 usable training rows


## Cell 3 — Optional: Weights & Biases logging

Skip this cell if you don't want experiment tracking, but I strongly recommend logging. Free for academic use. Run once to log in; the API key is stored in `~/.netrc` for the session.

In [6]:
import wandb

# This will prompt for your API key the first time. Get one at https://wandb.ai/authorize
# To skip W&B entirely, set USE_WANDB = False below.
USE_WANDB = True

if USE_WANDB:
    wandb.login()  # paste API key when prompted

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 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


wandb: WARNING Invalid choice


wandb: Enter your choice: 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


wandb: WARNING Invalid choice


...

wandb: WARNING Invalid choice


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vivekchandra726 (vivekchandra726-university-of-stirling) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Cell 4 — Training configuration

**Hyperparameters chosen for a ~270-image style LoRA on SDXL.** You can edit any of these — sensible ranges noted in comments. The defaults are conservative; if you find the result too weak after 2000 steps, increase `MAX_TRAIN_STEPS` to 3000 or raise `LEARNING_RATE` to 1.5e-4.

In [7]:
import torch

MODEL_NAME = "stabilityai/stable-diffusion-xl-base-1.0"
VAE_NAME = "madebyollin/sdxl-vae-fp16-fix"  # FP16-stable VAE — avoids NaN issues
OUTPUT_DIR = "/content/lora_output"

# Core hyperparams
LORA_RANK = 16              # 8 = lighter, 32 = more capacity. 16 is a good starting point.
LORA_ALPHA = 16             # usually = rank
LEARNING_RATE = 1e-4        # 5e-5 conservative, 1e-4 standard, 1.5e-4 aggressive
MAX_TRAIN_STEPS = 2000      # 1500–3000 typical; ~7–14 epochs over 270 images at batch 1
BATCH_SIZE = 1              # 6GB-friendly. On A100 you can go to 2 or 4.
GRADIENT_ACCUMULATION = 4   # effective batch = BATCH_SIZE * GRADIENT_ACCUMULATION
RESOLUTION = 1024           # SDXL's native; drop to 768 if VRAM-tight on smaller GPUs
CHECKPOINT_STEPS = 500      # save every N steps
LR_WARMUP_STEPS = 100
LR_SCHEDULER = "cosine"
MIXED_PRECISION = "bf16"    # bf16 on A100/L4; fall back to fp16 on T4 (cell below detects)
SEED = 42

# Detect GPU and adjust precision if necessary
gpu = torch.cuda.get_device_name(0)
if any(name in gpu for name in ["T4", "P100", "V100"]):
    MIXED_PRECISION = "fp16"
    print(f"Detected {gpu} — using fp16 (no bf16 support)")
else:
    print(f"Detected {gpu} — using bf16")

print(f"\nConfig summary:")
print(f"  rank/alpha:   {LORA_RANK}/{LORA_ALPHA}")
print(f"  steps:        {MAX_TRAIN_STEPS}")
print(f"  effective bs: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  resolution:   {RESOLUTION}")
print(f"  lr / warmup:  {LEARNING_RATE} / {LR_WARMUP_STEPS}")

Detected NVIDIA A100-SXM4-40GB — using bf16

Config summary:
  rank/alpha:   16/16
  steps:        2000
  effective bs: 4
  resolution:   1024
  lr / warmup:  0.0001 / 100


## Cell 5 — Download the diffusers training script

Rather than reimplementing a SDXL LoRA training loop (~400 lines of careful code), we use the official diffusers `train_text_to_image_lora_sdxl.py` script. This is the same script used in HuggingFace's LoRA tutorials and is battle-tested.

We pin to a specific diffusers commit for reproducibility.

In [8]:
# Download the official training script
!wget -q -O /content/train_text_to_image_lora_sdxl.py \
    https://raw.githubusercontent.com/huggingface/diffusers/v0.30.0/examples/text_to_image/train_text_to_image_lora_sdxl.py

!ls -la /content/train_text_to_image_lora_sdxl.py
!head -20 /content/train_text_to_image_lora_sdxl.py

-rw-r--r-- 1 root root 55568 Jun  3 19:44 /content/train_text_to_image_lora_sdxl.py
#!/usr/bin/env python
# coding=utf-8
# Copyright 2024 The HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
"""Fine-tuning script for Stable Diffusion XL for text2image with support for LoRA."""

import argparse
import logging
import math


## Cell 6 — Pre-flight: configure accelerate

`accelerate` handles the mixed-precision training setup. We need a config file before launching.

In [9]:
accelerate_config = f"""\
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: {MIXED_PRECISION}
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
"""

Path("/root/.cache/huggingface/accelerate").mkdir(parents=True, exist_ok=True)
Path("/root/.cache/huggingface/accelerate/default_config.yaml").write_text(accelerate_config)
print("Accelerate config written.")
!cat /root/.cache/huggingface/accelerate/default_config.yaml

Accelerate config written.
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false


## Cell 7 — Launch training

This is the long-running cell. Keep the Colab tab open and active. Checkpoints saved every 500 steps to both `OUTPUT_DIR` (Colab disk) and `DRIVE_LORA_OUT` (Google Drive) so a disconnect doesn't lose progress.

**Expected runtime:**
- A100 40GB: ~90–120 minutes for 2000 steps at resolution 1024
- L4 24GB: ~3–4 hours
- T4 16GB: 6+ hours (consider dropping RESOLUTION to 768 and BATCH_SIZE to 1)

In [11]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 110.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [12]:
# Construct the training command.
# Note: --train_data_dir points at the folder containing metadata.jsonl + image files.
# The diffusers script automatically detects that format.

import os
os.environ["WANDB_PROJECT"] = "msc-folk-art-packaging"
os.environ["WANDB_RUN_NAME"] = f"sdxl_lora_r{LORA_RANK}_lr{LEARNING_RATE}_steps{MAX_TRAIN_STEPS}"

report_to = "wandb" if USE_WANDB else "tensorboard"

cmd = f"""accelerate launch /content/train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path={MODEL_NAME} \
  --pretrained_vae_model_name_or_path={VAE_NAME} \
  --train_data_dir={LOCAL_IMAGES} \
  --caption_column=text \
  --resolution={RESOLUTION} \
  --center_crop \
  --random_flip \
  --train_batch_size={BATCH_SIZE} \
  --gradient_accumulation_steps={GRADIENT_ACCUMULATION} \
  --gradient_checkpointing \
  --max_train_steps={MAX_TRAIN_STEPS} \
  --learning_rate={LEARNING_RATE} \
  --lr_scheduler={LR_SCHEDULER} \
  --lr_warmup_steps={LR_WARMUP_STEPS} \
  --mixed_precision={MIXED_PRECISION} \
  --rank={LORA_RANK} \
  --use_8bit_adam \
  --checkpointing_steps={CHECKPOINT_STEPS} \
  --output_dir={OUTPUT_DIR} \
  --seed={SEED} \
  --report_to={report_to} \
  --validation_prompt="a photograph of {TRIGGER}, an Indian snack packet, vibrant, product photo on a white background" \
  --num_validation_images=2 \
  --validation_epochs=1
"""

print("Launching training. Expect first ~60 seconds to be model downloads.\n")
print(cmd)
print("\n" + "=" * 70)

!{cmd}

Launching training. Expect first ~60 seconds to be model downloads.

accelerate launch /content/train_text_to_image_lora_sdxl.py   --pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0   --pretrained_vae_model_name_or_path=madebyollin/sdxl-vae-fp16-fix   --train_data_dir=/content/training_images   --caption_column=text   --resolution=1024   --center_crop   --random_flip   --train_batch_size=1   --gradient_accumulation_steps=4   --gradient_checkpointing   --max_train_steps=2000   --learning_rate=0.0001   --lr_scheduler=cosine   --lr_warmup_steps=100   --mixed_precision=bf16   --rank=16   --use_8bit_adam   --checkpointing_steps=500   --output_dir=/content/lora_output   --seed=42   --report_to=wandb   --validation_prompt="a photograph of ipsnackpkg, an Indian snack packet, vibrant, product photo on a white background"   --num_validation_images=2   --validation_epochs=1


06/03/2026 19:49:48 - INFO - __main__ - [RANK 0] Distributed environment: DistributedType.NO
Num pro

## Cell 8 — Copy the final LoRA to Drive

After training completes, the LoRA weights are in `OUTPUT_DIR` on the Colab disk. We copy to Drive so you can download to your laptop and disconnect Colab without losing them.

In [13]:
import shutil

# The training script writes pytorch_lora_weights.safetensors in OUTPUT_DIR
src = Path(OUTPUT_DIR) / "pytorch_lora_weights.safetensors"
if not src.exists():
    # Look for the most recent checkpoint
    ckpts = sorted(Path(OUTPUT_DIR).glob("checkpoint-*/pytorch_lora_weights.safetensors"))
    if ckpts:
        src = ckpts[-1]
        print(f"Final weights not found; using last checkpoint: {src}")
    else:
        raise FileNotFoundError(f"No LoRA weights found in {OUTPUT_DIR}")

# Versioned filename
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M")
dst = DRIVE_LORA_OUT / f"sdxl_packaging_lora_r{LORA_RANK}_lr{LEARNING_RATE}_steps{MAX_TRAIN_STEPS}_{stamp}.safetensors"

shutil.copy2(src, dst)
print(f"LoRA copied to Drive: {dst}")
print(f"Size: {dst.stat().st_size / 1e6:.1f} MB")

# Also copy training config JSON for provenance
config = {
    "trigger": TRIGGER,
    "caption_template": CAPTION_TEMPLATE,
    "model": MODEL_NAME, "vae": VAE_NAME,
    "rank": LORA_RANK, "alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE, "steps": MAX_TRAIN_STEPS,
    "batch_size": BATCH_SIZE, "grad_accum": GRADIENT_ACCUMULATION,
    "resolution": RESOLUTION, "precision": MIXED_PRECISION,
    "scheduler": LR_SCHEDULER, "warmup": LR_WARMUP_STEPS,
    "seed": SEED, "timestamp": stamp,
    "n_train_images": len(list(LOCAL_IMAGES.glob('*.png'))),
}
config_path = dst.with_suffix(".json")
config_path.write_text(json.dumps(config, indent=2))
print(f"Config: {config_path}")

LoRA copied to Drive: /content/drive/MyDrive/dissertation/outputs/lora_checkpoints/sdxl_packaging_lora_r16_lr0.0001_steps2000_20260603_2249.safetensors
Size: 46.6 MB
Config: /content/drive/MyDrive/dissertation/outputs/lora_checkpoints/sdxl_packaging_lora_r16_lr0.0001_steps2000_20260603_2249.json


## Cell 9 — Inline validation: test the trained LoRA

Load SDXL, apply your fresh LoRA, generate 4 sample images using the trigger token. This tells you immediately whether the LoRA learned something meaningful before you go to the trouble of downloading and integrating it locally.

In [14]:
import torch, gc
from diffusers import StableDiffusionXLPipeline
from PIL import Image
import matplotlib.pyplot as plt

# Free training memory
gc.collect(); torch.cuda.empty_cache()

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
).to("cuda")

pipe.load_lora_weights(str(dst))
print("LoRA loaded.")

test_prompts = [
    f"a photograph of {TRIGGER}, an Indian snack packet, vibrant, product photo on a white background",
    f"a photograph of {TRIGGER}, masala flavour, professional product photography",
    f"a photograph of {TRIGGER}, red and yellow packaging, isolated on white",
    f"a photograph of {TRIGGER}, premium snack pack, studio lighting",
]

images = []
for i, p in enumerate(test_prompts):
    g = torch.Generator(device="cuda").manual_seed(42 + i)
    img = pipe(
        prompt=p,
        negative_prompt="blurry, low quality, watermark, distorted, deformed",
        num_inference_steps=30,
        guidance_scale=7.5,
        width=1024, height=1024,
        generator=g,
    ).images[0]
    images.append((p, img))

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, (p, im) in zip(axes.flat, images):
    ax.imshow(im); ax.axis("off")
    ax.set_title(p[:60] + "...", fontsize=8)
plt.tight_layout()
plt.savefig("/content/lora_validation_grid.png", dpi=120)
plt.show()

# Save the validation grid to Drive too
shutil.copy2("/content/lora_validation_grid.png", DRIVE_LORA_OUT / f"validation_{stamp}.png")
print(f"Validation grid saved to Drive: {DRIVE_LORA_OUT}/validation_{stamp}.png")

Output hidden; open in https://colab.research.google.com to view.

## What to do after this notebook finishes

1. **Look at the validation grid.** Do the four images look like Indian snack packets? Or like generic product shots? If the trigger token clearly invokes a learned domain (specific colours, layouts, conventions different from baseline SDXL), training succeeded. If they look identical to baseline SDXL output, training didn't learn — re-run with `LEARNING_RATE=1.5e-4` and `MAX_TRAIN_STEPS=3000`.
2. **Download the .safetensors file** from Drive to your laptop (`outputs/lora_checkpoints/`).
3. **Disconnect the Colab runtime** to stop billing.
4. **Next step locally:** combine LoRA + IP-Adapter-Plus in the existing inference pipeline, regenerate the spike's grid with both components active, and compare to Plus-only results.

**Common issues:**
- *Validation images look identical to baseline SDXL*: LR too low or steps too few. Retrain with higher.
- *Validation images look broken/melted*: LR too high. Retrain with LR=5e-5.
- *Validation images are over-saturated/identical to each other*: rank too high or overfitting. Try rank=8.
- *Validation images ignore the trigger token*: tokenizer didn't register the new token. Check the printed accelerate logs for tokenizer warnings.